# EBM-NLP Task 1: Preliminary Clustering Analysis

This notebook explores whether clinical trial text forms natural semantic clusters that align with the structured schema fields in the EBM-NLP task, namely **Participants (P)**, **Interventions (I)**, and **Outcomes (O)**.

The purpose of this stage is not to solve the extraction task directly, but to investigate whether clustering can reveal useful structure before building the main extraction pipeline.

## Section 1: Imports

In [3]:
## Importing the Liabraries  .....

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import umap
import plotly.express as px

from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA

## Section 2: Initial exploratory analysis using .txt files

In this step, we read a subset of clinical trial abstracts from the dataset and split them into sentences.

In [4]:
folder_path = r"C:\Users\vikra\OneDrive\Desktop\Study_Board\Bristol\SEM 2\Introduction to AI and Text Analytics\AITA Coursework\EBM-NLP-master\ebm_nlp_1_00\documents"
all_sentences = []
doc_ids = []

max_files = 2000  # to avoid overloading  
file_count = 0

for file in os.listdir(folder_path):
    if file.endswith(".text"):   # correct extension
        file_path = os.path.join(folder_path, file)

        with open(file_path, encoding="utf-8", errors="ignore") as f:
            text = f.read().strip()

        # simple sentence segmentation
        sentences = re.split(r'(?<=[.!?])\s+', text)

        # removing extra spaces
        sentences = [s.strip() for s in sentences if s.strip()]

        all_sentences.extend(sentences)
        doc_ids.extend([file] * len(sentences))

        file_count += 1
        if file_count >= max_files:
            break

print("Files processed:", file_count)
print("Total sentences:", len(all_sentences))
print("Unique files:", len(set(doc_ids)))
print("\nSample sentences:")
for s in all_sentences[:5]:
    print("-", s)

Files processed: 2000
Total sentences: 22752
Unique files: 2000

Sample sentences:
- [Triple therapy regimens involving H2 blockaders for therapy of Helicobacter pylori infections].
- Comparison of ranitidine and lansoprazole in short-term low-dose triple therapy for Helicobacter pylori infection.
- To evaluate the efficacy and safety of two 1-week low-dose triple-therapy drug regimens involving antisecretory drugs for Helicobacter pylori infection, 99 patients with H.
- pylori infection were treated with either lansoprazole (LPZ) or ranitidine (RNT) used together with clarithromycin (CAM) and metrinidazole (MTZ).
- The drug combination and administration periods in the PPI group were LPZ 30 mg, CAM 400 mg, MTZ 500 mg (LCM group).


### Observation

A large number of sentences were extracted from the selected abstracts, giving us a broad sample of the dataset for exploratory analysis.

The sample sentences show that abstracts contain a mixture of background, treatment descriptions, patient descriptions, and study outcomes. This already suggests that semantic content may be heterogeneous and not cleanly separable.

## Section 3: Sentence Embeddings

Here, each sentence is converted into a dense numerical vector using the pre-trained transformer model `all-MiniLM-L6-v2`.

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(all_sentences, show_progress_bar=True)

print("Embeddings shape:", embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/711 [00:00<?, ?it/s]

### Interpretation

The output shape shows that each sentence has been mapped to a fixed-length embedding vector.

For example, if the embedding matrix has shape `(22752, 384)`, this means:

- 22,752 sentences were processed
- each sentence is represented by 384 semantic features

This vector representation is the input used by the clustering algorithms in the next stage.

## Section 4: K means Clustering ....

The structured extraction task focuses on three main schema fields:

- Participants
- Interventions
- Outcomes

We therefore test whether sentence embeddings naturally separate into three broad semantic groups that might correspond to these fields.

### Important note
K-Means is unsupervised. It does not know what P, I, or O mean. It only groups points based on geometric similarity in embedding space.

In [ ]:
from sklearn.preprocessing import normalize
embeddings = normalize(embeddings)


kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(embeddings)

print("Total cluster labels:", len(cluster_labels))
print("Unique clusters:", np.unique(cluster_labels))

In [ ]:
df = pd.DataFrame({
    "sentence": all_sentences,
    "cluster": cluster_labels,
    "doc_id": doc_ids
})

df.sample(10, random_state=42)

### Initial Cluster Inspection

To understand what the clusters represent, we inspect a few example sentences from each cluster.

This qualitative inspection helps us judge whether the clusters appear coherent and whether they loosely resemble schema-related categories.

### Early impression
Some clusters appear to group semantically related sentences, but the categories are not cleanly interpretable as pure Participants, Interventions, or Outcomes. Many sentences contain mixed information.

## Section 5: PCA visualization

We reduce the high-dimensional sentence embeddings to two dimensions using PCA for plotting.

### Why PCA?
The original embeddings are high-dimensional and cannot be directly visualised. PCA provides a low-dimensional approximation that helps us inspect the overall cluster structure.

In [ ]:
pca = PCA(n_components=2)
embeddings_2d_pca = pca.fit_transform(embeddings)

plt.figure(figsize=(10, 7))
plt.scatter(
    embeddings_2d_pca[:, 0],
    embeddings_2d_pca[:, 1],
    c=cluster_labels,
    s=10
)

plt.title("KMeans Clustering of Sentence Embeddings (PCA Projection)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.show()

## Section 6: UMAP Visualization

In addition to PCA, we use UMAP to visualise the embedding space.

### Goal
To obtain a more informative visual view of how the sentence clusters are distributed.

In [ ]:
np.random.seed(42)
plot_idx = np.random.choice(len(embeddings), size=3000, replace=False)

embeddings_sample = embeddings[plot_idx]
cluster_sample = cluster_labels[plot_idx]

reducer = umap.UMAP(n_components=2, random_state=42)
embeddings_2d_umap = reducer.fit_transform(embeddings_sample)

plt.figure(figsize=(10, 7))
scatter = plt.scatter(
    embeddings_2d_umap[:, 0],
    embeddings_2d_umap[:, 1],
    c=cluster_sample,
    s=12
)

plt.title("KMeans Clustering of Sentence Embeddings (UMAP Projection)")
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.colorbar(scatter, label="Cluster")
plt.show()

## Section 7: Applying Hierarchical Clustering

This section applies **Hierarchical Agglomerative Clustering** to a subset of the sentence embeddings.

### Why compare HAC with K-Means?
K-Means assumes roughly spherical clusters and partitions the space directly. HAC builds clusters by progressively merging nearby points, which may capture different structure.

Comparing both methods helps us test whether the observed clustering pattern is robust across different unsupervised approaches.

In [ ]:
np.random.seed(42)
idx = np.random.choice(len(embeddings), size=2000)

embeddings_subset = embeddings[idx]
sentences_subset = [all_sentences[i] for i in idx]

hac = AgglomerativeClustering(n_clusters=3, linkage='ward')
hac_labels = hac.fit_predict(embeddings_subset)

In [ ]:
import umap
import matplotlib.pyplot as plt

reducer = umap.UMAP(n_components=2, random_state=20)
embeddings_subset_2d = reducer.fit_transform(embeddings_subset)

plt.figure(figsize=(10,7))
plt.scatter(embeddings_subset_2d[:,0], embeddings_subset_2d[:,1], c=hac_labels)
plt.title("HAC on Sentence Embeddings (Subset)")
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.show()

## 3D umap visualization plot (for reference only)

In [ ]:
# random subset
np.random.seed(42)
plot_idx = np.random.choice(len(embeddings), size=3000, replace=False)

embeddings_sample = embeddings[plot_idx]
cluster_sample = cluster_labels[plot_idx]

# UMAP 3D
reducer_3d = umap.UMAP(n_components=3, random_state=42)
umap_result_3d = reducer_3d.fit_transform(embeddings_sample)

sample_sentences = [all_sentences[i] for i in plot_idx]

# create plot
fig = px.scatter_3d(
    x=umap_result_3d[:, 0],
    y=umap_result_3d[:, 1],
    z=umap_result_3d[:, 2],
    color=cluster_sample.astype(str),
    title="3D UMAP Projection of Sentence Embedding Clusters",
    labels={"color": "Cluster"},
    hover_data={"sentence": sample_sentences}
)

fig.update_layout(
    width=1200,
    height=900,
    margin=dict(l=60, r=60, b=60)
)

fig.update_layout(
    scene=dict(
        xaxis_title='UMAP 1',
        yaxis_title='UMAP 2',
        zaxis_title='UMAP 3'
    )
)

fig.show()

# =====================================
# Section 9: Sentence Label Mapping (Using Annotations)
# =====================================

So far, clustering has been completely unsupervised. However, to evaluate whether the clusters correspond to meaningful schema fields, we need a reference label for each sentence.

The EBM-NLP dataset provides token-level annotations for:

- Participants
- Interventions
- Outcomes

### What we do here
We map token-level annotations to sentence-level labels by checking which schema field appears most often within each sentence.

### Why is this needed?
Cluster IDs such as `0`, `1`, and `2` do not have intrinsic meaning. To evaluate clustering quality, we must compare them with gold labels derived from the dataset annotations.

In [ ]:
import os

base_path = r"C:\Users\vikra\OneDrive\Desktop\Study_Board\Bristol\SEM 2\Introduction to AI and Text Analytics\AITA Coursework\EBM-NLP-master\ebm_nlp_1_00"

documents_path = os.path.join(base_path, "documents")
ann_base = os.path.join(base_path, "annotations", "aggregated", "hierarchical_labels")

participants_path = os.path.join(ann_base, "participants", "train")
interventions_path = os.path.join(ann_base, "interventions", "train")
outcomes_path = os.path.join(ann_base, "outcomes", "train")

In [ ]:
def read_text_file(file_path):
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read().strip()

def read_tokens_file(file_path):
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read().strip().split()

def read_labels_file(file_path):
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read().strip().split(",")

In [ ]:
pmid = "43164"

text_file = os.path.join(documents_path, f"{pmid}.text")
tokens_file = os.path.join(documents_path, f"{pmid}.tokens")

participants_file = os.path.join(participants_path, f"{pmid}_AGGREGATED.ann")
interventions_file = os.path.join(interventions_path, f"{pmid}_AGGREGATED.ann")
outcomes_file = os.path.join(outcomes_path, f"{pmid}_AGGREGATED.ann")

### Data Alignment Check

Before creating sentence-level labels, we verify that:

- the token sequence length
- participant label length
- intervention label length
- outcome label length

all match correctly.

This is essential because sentence-level gold labels can only be trusted if the underlying token-level annotations are aligned with the tokenised abstract.

In [ ]:
text = read_text_file(text_file)
tokens = read_tokens_file(tokens_file)

p_labels = read_labels_file(participants_file)
i_labels = read_labels_file(interventions_file)
o_labels = read_labels_file(outcomes_file)

# ---------- Debug ----------

print("Text length:", len(text))
print("Number of tokens:", len(tokens))
print("Participant labels:", len(p_labels))
print("Intervention labels:", len(i_labels))
print("Outcome labels:", len(o_labels))

In [ ]:
print("First 20 tokens:")
print(tokens[:20])

print("\nFirst 20 participant labels:")
print(p_labels[:20])

print("\nFirst 20 intervention labels:")
print(i_labels[:20])

print("\nFirst 20 outcome labels:")
print(o_labels[:20])

In [ ]:
def read_tokens_file(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read().strip()
    return text.split()

def read_label_file(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read().strip()
    return [int(x) for x in text.split(",") if x.strip() != ""]

In [ ]:
pmid = "43164"

text_file = os.path.join(documents_path, f"{pmid}.text")
tokens_file = os.path.join(documents_path, f"{pmid}.tokens")

participants_file = os.path.join(participants_path, f"{pmid}_AGGREGATED.ann")
interventions_file = os.path.join(interventions_path, f"{pmid}_AGGREGATED.ann")
outcomes_file = os.path.join(outcomes_path, f"{pmid}_AGGREGATED.ann")

text = read_text_file(text_file)
tokens = read_tokens_file(tokens_file)
p_labels = read_label_file(participants_file)
i_labels = read_label_file(interventions_file)
o_labels = read_label_file(outcomes_file)

print("Text length:", len(text))
print("Number of tokens:", len(tokens))
print("Participant labels:", len(p_labels))
print("Intervention labels:", len(i_labels))
print("Outcome labels:", len(o_labels))

In [ ]:
from collections import Counter

def tokens_to_sentence_labels(tokens, p_labels, i_labels, o_labels):
    sentences = []
    sentence_true_labels = []

    current_tokens = []
    current_labels = []

    for tok, p, i, o in zip(tokens, p_labels, i_labels, o_labels):
        current_tokens.append(tok)

        # collect schema hits
        if p > 0:
            current_labels.append("P")
        if i > 0:
            current_labels.append("I")
        if o > 0:
            current_labels.append("O")

        # sentence boundary
        if tok in [".", "!", "?"]:
            sentence_text = " ".join(current_tokens).strip()

            if len(current_labels) == 0:
                sentence_label = None
            else:
                counts = Counter(current_labels)
                sentence_label = counts.most_common(1)[0][0]

            sentences.append(sentence_text)
            sentence_true_labels.append(sentence_label)

            current_tokens = []
            current_labels = []

    # leftover tokens if any
    if current_tokens:
        sentence_text = " ".join(current_tokens).strip()

        if len(current_labels) == 0:
            sentence_label = None
        else:
            counts = Counter(current_labels)
            sentence_label = counts.most_common(1)[0][0]

        sentences.append(sentence_text)
        sentence_true_labels.append(sentence_label)

    return sentences, sentence_true_labels

In [ ]:
sentences_all = []
true_labels_all = []
doc_ids_all = []

max_files = 1000
file_count = 0

for file in os.listdir(documents_path):
    if file.endswith(".tokens"):
        pmid = file.replace(".tokens", "")

        tokens_file = os.path.join(documents_path, f"{pmid}.tokens")
        participants_file = os.path.join(participants_path, f"{pmid}_AGGREGATED.ann")
        interventions_file = os.path.join(interventions_path, f"{pmid}_AGGREGATED.ann")
        outcomes_file = os.path.join(outcomes_path, f"{pmid}_AGGREGATED.ann")

        # skip if any annotation file missing
        if not (
            os.path.exists(participants_file)
            and os.path.exists(interventions_file)
            and os.path.exists(outcomes_file)
        ):
            continue

        tokens = read_tokens_file(tokens_file)
        p_labels = read_label_file(participants_file)
        i_labels = read_label_file(interventions_file)
        o_labels = read_label_file(outcomes_file)

        # ensure alignment
        if not (len(tokens) == len(p_labels) == len(i_labels) == len(o_labels)):
            continue

        file_sentences, file_true_labels = tokens_to_sentence_labels(
            tokens, p_labels, i_labels, o_labels
        )

        sentences_all.extend(file_sentences)
        true_labels_all.extend(file_true_labels)
        doc_ids_all.extend([pmid] * len(file_sentences))

        file_count += 1
        if file_count >= max_files:
            break

print("Files processed:", file_count)
print("Total sentences:", len(sentences_all))
print("Total true labels:", len(true_labels_all))
print("Total doc ids:", len(doc_ids_all))

In [ ]:
filtered_sentences = []
filtered_true_labels = []
filtered_doc_ids = []

for s, lbl, doc_id in zip(sentences_all, true_labels_all, doc_ids_all):
    if lbl is not None:
        filtered_sentences.append(s)
        filtered_true_labels.append(lbl)
        filtered_doc_ids.append(doc_id)

print("Filtered sentences:", len(filtered_sentences))
print("Filtered true labels:", len(filtered_true_labels))

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(filtered_sentences, show_progress_bar=True)

print("Embeddings shape:", embeddings.shape)

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(embeddings)

print("Total cluster labels:", len(cluster_labels))
print("Unique clusters:", np.unique(cluster_labels))

In [ ]:
df = pd.DataFrame({
    "sentence": filtered_sentences,
    "cluster": cluster_labels,
    "true_label": filtered_true_labels,
    "doc_id": filtered_doc_ids
})

df.head()

In [ ]:
ct = pd.crosstab(df["cluster"], df["true_label"])
print(ct)

In [ ]:
ct.plot(kind="bar", stacked=True, figsize=(8, 5))
plt.title("Distribution of True Labels within Clusters")
plt.xlabel("Cluster")
plt.ylabel("Count")
plt.show()

In [ ]:
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(embeddings)

plt.figure(figsize=(10, 7))
plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], c=cluster_labels, s=10)
plt.title("KMeans Clustering of Sentence Embeddings (PCA Projection)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

In [ ]:
np.random.seed(42)
plot_idx = np.random.choice(len(embeddings), size=min(3000, len(embeddings)), replace=False)

embeddings_sample = embeddings[plot_idx]
cluster_sample = cluster_labels[plot_idx]

reducer = umap.UMAP(n_components=2, random_state=42)
embeddings_2d_umap = reducer.fit_transform(embeddings_sample)

plt.figure(figsize=(10, 7))
plt.scatter(embeddings_2d_umap[:, 0], embeddings_2d_umap[:, 1], c=cluster_sample, s=12)
plt.title("UMAP Projection of Sentence Embedding Clusters")
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.show()

In [ ]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

label_map = {"P": 0, "I": 1, "O": 2}
true_numeric = [label_map[x] for x in filtered_true_labels]

ari = adjusted_rand_score(true_numeric, cluster_labels)
nmi = normalized_mutual_info_score(true_numeric, cluster_labels)

print("ARI:", ari)
print("NMI:", nmi)